In [5]:

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
        
####### FORCE CPU #######
# import os
# # This must be set BEFORE importing tensorflow
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# import tensorflow as tf

# # Optional: Verify that no GPUs are visible
# print("GPUs visible to TensorFlow:", tf.config.list_physical_devices('GPU'))
# # This should print: GPUs visible to TensorFlow: []

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Exo1.1:

#### 1.

In [6]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras import Input
from keras.layers import Dropout

# load dataset
dataset = pd.read_csv( 'Churn_Modelling.csv')
X = dataset.iloc [: , 3:13]. values
y = dataset.iloc [: , 13]. values


# Preprocess categorical variables
labelencoder_X_1 = LabelEncoder ()
X [: , 1]=labelencoder_X_1.fit_transform ( X [: , 1])
labelencoder_X_2 = LabelEncoder ()
X [: , 2]=labelencoder_X_2.fit_transform ( X [: , 2])


# One - hot encoding for country
ct = ColumnTransformer ([( "Geography" , OneHotEncoder () , [1]) ] ,remainder = 'passthrough')
X = ct . fit_transform ( X )
X = X [: , 1:] # Avoid dummy variable trap


# Split data
X_train , X_test , y_train , y_test = train_test_split (X , y , test_size =0.2 , random_state =0)


 # Feature scaling
sc = StandardScaler ()
X_train = sc.fit_transform ( X_train )
X_test = sc.transform ( X_test )



In [7]:

# Build ANN
classifier = Sequential ()
classifier.add(Input(shape=(11,)))
classifier . add ( Dense ( units =6 , activation = 'relu' ) )
classifier . add ( Dense ( units =6 , activation = 'relu') )
classifier . add ( Dense ( units =1 , activation = 'sigmoid') )


# Compile
classifier . compile ( optimizer ='adam', loss ='binary_crossentropy', metrics =['accuracy'])


# Train
classifier . fit ( X_train , y_train , batch_size =10 , epochs =100)


# Predict
y_pred = classifier . predict ( X_test )
y_pred = ( y_pred > 0.5)


# Evaluate
from sklearn . metrics import confusion_matrix
cm = confusion_matrix ( y_test , y_pred )
print ( "Confusion Matrix :\n" , cm )

Epoch 1/100
800/800 [==============================] - 3s 4ms/step - loss: 0.5456 - accuracy: 0.7887
Epoch 2/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4634 - accuracy: 0.7965
Epoch 3/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4334 - accuracy: 0.8024
Epoch 4/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4245 - accuracy: 0.8089
Epoch 5/100
800/800 [==============================] - 3s 3ms/step - loss: 0.4203 - accuracy: 0.8130
Epoch 6/100
800/800 [==============================] - 2s 2ms/step - loss: 0.4155 - accuracy: 0.8152
Epoch 7/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4100 - accuracy: 0.8186
Epoch 8/100
800/800 [==============================] - 3s 4ms/step - loss: 0.4025 - accuracy: 0.8176
Epoch 9/100
800/800 [==============================] - 2s 3ms/step - loss: 0.3933 - accuracy: 0.8155
Epoch 10/100
800/800 [==============================] - 2s 3ms/step - loss: 0.3794 - accura

##### comments:
relatively high accuracy 86% 

however the minority class (negative) has near 50% accuracy

#### 2.

In [8]:


# Build ANN
classifier = Sequential ()
classifier.add(Input(shape=(11,)))
classifier . add ( Dense ( units =6 , activation = 'relu' ) )
classifier . add ( Dense ( units =6 , activation = 'relu') )
classifier . add( Dropout(rate=0.2))
classifier . add ( Dense ( units =1 , activation = 'sigmoid') )


# Compile
classifier . compile ( optimizer ='adam', loss ='binary_crossentropy', metrics =['accuracy'])


# Train
classifier . fit ( X_train , y_train , batch_size =10 , epochs =100)


# Predict
y_pred = classifier . predict ( X_test )
y_pred = ( y_pred > 0.5)


# Evaluate
from sklearn . metrics import confusion_matrix
cm = confusion_matrix ( y_test , y_pred )
print ( "Confusion Matrix :\n" , cm )

Epoch 1/100
800/800 [==============================] - 2s 2ms/step - loss: 0.5274 - accuracy: 0.7735
Epoch 2/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4634 - accuracy: 0.7989
Epoch 3/100
800/800 [==============================] - 2s 2ms/step - loss: 0.4391 - accuracy: 0.8079
Epoch 4/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4301 - accuracy: 0.8152
Epoch 5/100
800/800 [==============================] - 2s 2ms/step - loss: 0.4232 - accuracy: 0.8211
Epoch 6/100
800/800 [==============================] - 2s 2ms/step - loss: 0.4175 - accuracy: 0.8230
Epoch 7/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4109 - accuracy: 0.8266
Epoch 8/100
800/800 [==============================] - 2s 3ms/step - loss: 0.4019 - accuracy: 0.8289
Epoch 9/100
800/800 [==============================] - 2s 2ms/step - loss: 0.3905 - accuracy: 0.8367
Epoch 10/100
800/800 [==============================] - 2s 3ms/step - loss: 0.3831 - accura

##### comments:
no improvement

#### 3.

In [9]:
import tensorflow as tf
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV

# 1. Function to create the model
def create_model(optimizer='adam'):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(units=6, activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(units=6, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 2. Wrap using the legacy Keras wrapper
# This version is built into TensorFlow 2.10
classifier = KerasClassifier(build_fn=create_model)

from tensorflow.keras.callbacks import EarlyStopping

# Define the monitor
early_stopping = EarlyStopping(
    monitor='accuracy',    # What to watch (can also be 'loss' or 'val_accuracy')
    patience=10,           # How many epochs to wait for improvement before stopping
    verbose=1,             # Tells you when it stops
    mode='max',            # We want accuracy to be at its maximum
    restore_best_weights=True # Very important! Reverts the model to its best state
)

# 3. Define parameters
parameters = {
    'batch_size': [ 25, 32],
    'epochs': [100],
    'optimizer': ['adam', 'rmsprop'],
    'callbacks':[early_stopping]
}

# 4. Initialize and Run
grid_search = GridSearchCV(estimator=classifier,
                           param_grid=parameters,
                           scoring='accuracy',)

grid_search = grid_search.fit(X_train, y_train)

Epoch 1/100


/tmp/ipykernel_110746/3446481086.py:18: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  classifier = KerasClassifier(build_fn=create_model)


256/256 [==============================] - 1s 3ms/step - loss: 0.6909 - accuracy: 0.6400
Epoch 2/100
256/256 [==============================] - 1s 3ms/step - loss: 0.5442 - accuracy: 0.7767
Epoch 3/100
256/256 [==============================] - 1s 2ms/step - loss: 0.5041 - accuracy: 0.7948
Epoch 4/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4712 - accuracy: 0.8037
Epoch 5/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4540 - accuracy: 0.8120
Epoch 6/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4412 - accuracy: 0.8175
Epoch 7/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4319 - accuracy: 0.8175
Epoch 8/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4274 - accuracy: 0.8230
Epoch 9/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4323 - accuracy: 0.8223
Epoch 10/100
256/256 [==============================] - 1s 3ms/step - loss: 0.4321 - accuracy: 0.8255
E

In [10]:
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_

print("--- Best Model Stats ---")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best Parameters: {best_parameters}")

--- Best Model Stats ---
Best Accuracy: 0.8577
Best Parameters: {'batch_size': 32, 'callbacks': <keras.callbacks.EarlyStopping object at 0x7fef20256d30>, 'epochs': 100, 'optimizer': 'adam'}


##### comments:
a smaller batch size and adam optimizer worked best in this case, early stopping was implemented to avoid wasting runtime, as it triggered many times using 100 epochs it would be a waste to test 200 epochs.
batch size 25 likely worked better because of the imbalace of the dataset.

## Exo1.2:

In [11]:
from keras . wrappers . scikit_learn import KerasClassifier
from sklearn . model_selection import cross_val_score

# Function to build the classifier
def build_classifier () :
    classifier = Sequential ()
    classifier . add ( Dense ( units =6 , activation ='relu', input_dim=11) )
    classifier . add ( Dense ( units =6 , activation ='relu') )
    classifier . add ( Dense ( units =1 , activation ='sigmoid') )
    classifier . compile ( optimizer ='adam', loss ='binary_crossentropy', metrics =['accuracy'])
    
    return classifier

# Create classifier
classifier = KerasClassifier ( build_fn = build_classifier ,
batch_size =25 , epochs =100)

# Perform cross - validation
accuracies = cross_val_score ( estimator = classifier , X = X_train , y =
y_train , cv =5)

# Results
print ( " Mean  accuracy : " , accuracies . mean () )
print ( " Standard  deviation : " , accuracies . std () )

Epoch 1/100


/tmp/ipykernel_110746/344804476.py:15: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  classifier = KerasClassifier ( build_fn = build_classifier ,


256/256 [==============================] - 1s 2ms/step - loss: 0.5062 - accuracy: 0.7980
Epoch 2/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4578 - accuracy: 0.7980
Epoch 3/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4433 - accuracy: 0.7980
Epoch 4/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4374 - accuracy: 0.7980
Epoch 5/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4342 - accuracy: 0.7980
Epoch 6/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4318 - accuracy: 0.7980
Epoch 7/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4294 - accuracy: 0.7980
Epoch 8/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4269 - accuracy: 0.7980
Epoch 9/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4241 - accuracy: 0.7980
Epoch 10/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4216 - accuracy: 0.7980
E

In [12]:
from keras . wrappers . scikit_learn import KerasClassifier
from sklearn . model_selection import cross_val_score

# Function to build the classifier
def build_classifier () :
    classifier = Sequential ()
    classifier . add ( Dense ( units =6 , activation ='relu', input_dim=11) )
    
    classifier . add ( Dense ( units =6 , activation ='relu') )
    classifier . add( Dropout(rate=0.2))
    classifier . add ( Dense ( units =1 , activation ='sigmoid') )
    classifier . compile ( optimizer ='adam', loss ='binary_crossentropy', metrics =['accuracy'])
    
    return classifier

# Create classifier
classifier = KerasClassifier ( build_fn = build_classifier ,
batch_size =25 , epochs =100)

# Perform cross - validation
accuracies = cross_val_score ( estimator = classifier , X = X_train , y =
y_train , cv =5)

# Results
print ( " Mean  accuracy : " , accuracies . mean () )
print ( " Standard  deviation : " , accuracies . std () )

Epoch 1/100


/tmp/ipykernel_110746/1012010013.py:17: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  classifier = KerasClassifier ( build_fn = build_classifier ,


256/256 [==============================] - 1s 2ms/step - loss: 0.5701 - accuracy: 0.7478
Epoch 2/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4896 - accuracy: 0.7980
Epoch 3/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4689 - accuracy: 0.8019
Epoch 4/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4485 - accuracy: 0.8086
Epoch 5/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4430 - accuracy: 0.8091
Epoch 6/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4391 - accuracy: 0.8183
Epoch 7/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4307 - accuracy: 0.8200
Epoch 8/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4273 - accuracy: 0.8198
Epoch 9/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4188 - accuracy: 0.8231
Epoch 10/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4188 - accuracy: 0.8253
E

##### comments:
dropout layers make the model more unstable (higher standard deviation)

## Exo1.3:

In [13]:
import tensorflow as tf
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV

# 1. Function to create the model
def create_model(optimizer='adam'):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(units=6, activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(units=6, activation='relu'),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 2. Wrap using the legacy Keras wrapper
# This version is built into TensorFlow 2.10
classifier = KerasClassifier(build_fn=create_model)

from tensorflow.keras.callbacks import EarlyStopping

# Define the monitor
early_stopping = EarlyStopping(
    monitor='accuracy',    # What to watch (can also be 'loss' or 'val_accuracy')
    patience=10,           # How many epochs to wait for improvement before stopping
    verbose=1,             # Tells you when it stops
    mode='max',            # We want accuracy to be at its maximum
    restore_best_weights=True # Very important! Reverts the model to its best state
)

# 3. Define parameters
parameters = {
    'batch_size': [ 25, 32],
    'epochs': [100, 200],
    'optimizer': ['adam', 'rmsprop'],
    'callbacks':[early_stopping]
}

# 4. Initialize and Run
grid_search = GridSearchCV(estimator=classifier,
                           param_grid=parameters,
                           scoring='accuracy',)

grid_search = grid_search.fit(X_train, y_train)

Epoch 1/100


/tmp/ipykernel_110746/2029386721.py:17: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  classifier = KerasClassifier(build_fn=create_model)


256/256 [==============================] - 1s 2ms/step - loss: 0.5047 - accuracy: 0.7936
Epoch 2/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4452 - accuracy: 0.8041
Epoch 3/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4313 - accuracy: 0.8128
Epoch 4/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4226 - accuracy: 0.8166
Epoch 5/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4169 - accuracy: 0.8223
Epoch 6/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4126 - accuracy: 0.8253
Epoch 7/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4087 - accuracy: 0.8273
Epoch 8/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4044 - accuracy: 0.8264
Epoch 9/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4004 - accuracy: 0.8295
Epoch 10/100
256/256 [==============================] - 1s 2ms/step - loss: 0.3943 - accuracy: 0.8355
E

In [14]:
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_

print("--- Best Model Stats ---")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best Parameters: {best_parameters}")

--- Best Model Stats ---
Best Accuracy: 0.8591
Best Parameters: {'batch_size': 25, 'callbacks': <keras.callbacks.EarlyStopping object at 0x7fef200cba90>, 'epochs': 200, 'optimizer': 'adam'}


In [15]:
import tensorflow as tf
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV

# 1. Function to create the model
def create_model(optimizer='adam'):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(units=6, activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(units=6, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(units=1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 2. Wrap using the legacy Keras wrapper
# This version is built into TensorFlow 2.10
classifier = KerasClassifier(build_fn=create_model)

from tensorflow.keras.callbacks import EarlyStopping

# Define the monitor
early_stopping = EarlyStopping(
    monitor='accuracy',    # What to watch (can also be 'loss' or 'val_accuracy')
    patience=10,           # How many epochs to wait for improvement before stopping
    verbose=1,             # Tells you when it stops
    mode='max',            # We want accuracy to be at its maximum
    restore_best_weights=True # Very important! Reverts the model to its best state
)

# 3. Define parameters
parameters = {
    'batch_size': [ 25, 32],
    'epochs': [100, 200],
    'optimizer': ['adam', 'rmsprop'],
    'callbacks':[early_stopping]
}

# 4. Initialize and Run
grid_search = GridSearchCV(estimator=classifier,
                           param_grid=parameters,
                           scoring='accuracy',)

grid_search = grid_search.fit(X_train, y_train)

Epoch 1/100


/tmp/ipykernel_110746/2660586579.py:19: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  classifier = KerasClassifier(build_fn=create_model)


256/256 [==============================] - 1s 2ms/step - loss: 0.6030 - accuracy: 0.7258
Epoch 2/100
256/256 [==============================] - 1s 2ms/step - loss: 0.5122 - accuracy: 0.7912
Epoch 3/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4912 - accuracy: 0.7961
Epoch 4/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4711 - accuracy: 0.8017
Epoch 5/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4627 - accuracy: 0.8041
Epoch 6/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4532 - accuracy: 0.8080
Epoch 7/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4528 - accuracy: 0.8048
Epoch 8/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4458 - accuracy: 0.8136
Epoch 9/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4399 - accuracy: 0.8141
Epoch 10/100
256/256 [==============================] - 1s 2ms/step - loss: 0.4397 - accuracy: 0.8152
E

In [16]:
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_

print("--- Best Model Stats ---")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best Parameters: {best_parameters}")

--- Best Model Stats ---
Best Accuracy: 0.8402
Best Parameters: {'batch_size': 32, 'callbacks': <keras.callbacks.EarlyStopping object at 0x7fef202408e0>, 'epochs': 100, 'optimizer': 'rmsprop'}


#### comments
dropout layers made the model more unstable (bigger batch size was better) and "agreed" more with rmsprop than adam because killing neurons adds momentum noise and makes adam updates noisy and inaccurate 